# Notebook de traitement des professions de foi par catégorisation
Ce notebook contient l'ensemble des fonctons que nous avons utilisées sous format prêt à l'emploi pour obtenir les résultats présentés.


In [45]:
from openai import OpenAI
import json
import os
import pandas as pd

In [ ]:
api_key = "" # Clé API à renseigner
MODEL = "mistralai/mistral-small-2603"
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")


In [55]:
import json

def run_mistral(system_prompt, client, content):
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": json.dumps(content, ensure_ascii=False)  # serialize to string
        }
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    return response.choices[0].message.content

## Classification
Mistral models can easily categorize text into distinct classes. In this example prompt, we can define a list of predefined categories and ask Mistral models to classify user inquiry.



In [48]:
# Fonction de définition du prompt avec le prompt spécifié directement.
# L'accent est mis sur le format, bien que ça ne semble pas avoir beaucoup d'effet.
# Dans l'ensemble, nous avons listé les consignes indiquées dans les tableaux descriptifs de la base de donnée sous la forme (catégorie | exemple | consignes).
# Dans une première version du prompt, nous avions précisé cette logique de représentation à l'intérieur du prompt. Il pourrait être pertinent
# de tenter de la remettre dedans pour voir si les résultats sont meilleurs en termes de format.
# Enfin, nous indiquons ce à quoi ressemble le fichier d'origine menant aux exemples explicités au-dessus pour qu'il puisse s'y référer.
def user_message():
    try:
      with open("/Users/charles-/Downloads/Hackathon/SciencesPo/Présentation/updatePrompt_final_SciencesPo.txt", "r") as prompt_file:
          user_message = prompt_file.read()
    except:
      print("Erreur dans la lecture du prompt!")
      return ""
    return user_message

In [49]:
from pathlib import Path
import pandas as pd

# Cellule de chargement des fichiers (sous forme de dataframes) dans la liste batches
# Quand lancée, elle doit print "Loaded batch_test_x.csv with 11 rows" 50 fois et après "50 dataframes in total" s'il y a 50 fichiers de 11 lignes.
# C'est normal si les fichiers traités ne le sont pas dans l'ordre. Mais cela signifie qu'il faut prendre en compte que le fichier 0 dans la liste
# n'est pas le fichier 0 en absolu. Se référer aux affichages pour identifier l'emplacement d'un fichier spécifique.

folder = Path("/Users/charles-/Downloads/Hackathon/SciencesPo/") 
endFolder = "/Users/charles-/Downloads/Hackathon/SciencesPo/resultatsNewPrompt/"

batches = []

for f in folder.iterdir():
    if f.is_file() and f.suffix == ".csv":  # S'assurer que le fichier a une extension ".csv"
        try:
            df = pd.read_csv(f)
            print(f"Loaded {f.name} with {len(df)} rows")
            batches.append(df)
        except Exception as e:
            print(f"Skipping {f.name}: {e}")

print(f"{len(batches)} dataframes loaded in total")


Loaded fichier_echantillon_5_test_mistral.csv with 5 rows
1 dataframes loaded in total


In [50]:
batches[0]

,filename,page_1,page_2
0,LG17-1-1-BLATRIX-CONTAT-2-tour1-profession_foi...,Michel FONTAINE\r\nFlorence Votre Députée supp...,"NATIONALEMENT, je serai une députée constructi..."
1,LG17-1-1-BONNOT-10-tour1-profession_foi.pdf,Vincent GUERIN Gilbert BONNOT\r\nSuppléant Can...,Ce que nous voulons\r\nDans une société dégrad...
2,LG17-1-1-BRETON-5-tour1-profession_foi.pdf,"»\r\nxavier\r\nEngagé,\r\nbreton\r\nà vos côté...",–\r\n»\r\nJe m’engage Je m’engage\r\nà encoura...
3,LG17-1-1-BRETON-5-tour2-profession_foi.pdf,"»\r\nxavier\r\nEngagé,\r\nbreton\r\nà vos côté...",–\r\n• •\r\n•\r\n• •\r\n•\r\n•\r\n•\r\n•\r\n•\...
4,LG17-1-1-CARLIER-6-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES – 11 ET 18 JUIN 2017\r\...,• défendront notre industrie contre les déloca...


In [56]:
##################################### THE CELLULE #####################################
# Elle se run sur les batches (sous-fichiers de 11 individus) construits précédemment à partir du dossier issu du Google Drive connecté.
# Je pense que c'est mieux est de créer dossier "results" dans le dossier Colab Notebooks de façon à trier proprement, parce qu'il enregistre
# les résultats dedans.
answers_2 = []

for i, _batch in enumerate(batches):
  if (i == 0): # Ne lance que sur le premier fichier chargé, pour tester que cela fonctionne. (donc pas forcément le premier en ordre alphanumérique)

        _batch["merged_text"] = _batch.iloc[:, 1:].apply(lambda row: " ".join(row.dropna().astype(str)), axis=1)
        batch = _batch[["filename", "merged_text"]]

        lignes = batch.to_dict(orient="records")

        #print(lignes)
        var = run_mistral(user_message(), client=client, content=lignes)# mistral-large-latest ministral-3b-2512

        with open(endFolder + f"resultatsBatch{i}MistralSmall.txt", "w", encoding="utf-8") as f:
            f.write(var)

        answers_2.append(var)

        # Enregistre dans le fichier results sous le nom "resultsBatch_i.csv" pour i le numéro du batch.
        with open(os.path.join(endFolder, f"resultsBatch_{i}.json"), "w") as output_file:
            json.dump(var[8:-4], output_file)


In [27]:
print(lignes[1])
print(answers_2)

{'filename': 'LG17-1-1-BONNOT-10-tour1-profession_foi.pdf', 'merged_text': 'Vincent GUERIN Gilbert BONNOT\r\nSuppléant Candidat\r\nEducateur Spécialisé Moniteur Educateur\r\nELECTIONS LEGISLATIVES - 11 et 18 Juin 2017\r\n1ère Circonscription\r\nPourquoi notre candidature\r\nNotre volonté est de construire d’en bas, un solide contre pouvoir avec vous\r\nélecteurs de la première circonscription de l’Ain.\r\nComme vous, concernés par les fins de mois difficiles, les factures à payer, la\r\npression liée au travail et tout autre difficulté de la vie courante.\r\nComme vous, nous ne sommes pas ou peu entendus.\r\nPar ce choix de candidature, nous souhaitons ne plus subir la question politique\r\nmais se la réaproprier.\r\nNotre premier engagement sera de vous donner la parole par notre voix.\r\nNous ne voulons plus nous taire de façon à ce que vous soyez entendus.\r\nPourquoi voter pour nous\r\nEn nous apportant votre voix, retrouvez l’envie de résister et reprenez votre\r\nrôle d’acteur de

In [57]:
with open("/Users/charles-/Downloads/Hackathon/SciencesPo/resultatsNewPrompt/resultsBatch_0.json") as f:
    raw = json.load(f)  # loads the outer string

data = json.loads(raw)  # parses the inner JSON string into a list
df = pd.DataFrame(data)
df.to_csv("/Users/charles-/Downloads/Hackathon/SciencesPo/resultatsNewPrompt/resultsBatch_0.csv")

In [58]:
df

,dc:identifier,ba_nom_titulaire,bb_prenom_titulaire,bd_age_titulaire,bc_sexe_titulaire,be_professionn_titulaire,bf_mandat_en_cours_titulaire,bg_mandat_passe_titulaire,bh_associations_titulaire,bi_autres_statuts_titulaire,...,cj_soutien_suppleant,ck_liste_suppleant,bj_soutien_titulaire,bk_liste_titulaire,dc:title,dc:date,contexte_tour,nom_departement,departement,df_circonscription
0,LG17-1-1-BLATRIX-CONTAT-2-tour1-profession_foi...,Blatrix-Contat,Florence,51,F,enseignante,conseiller régional,conseiller municipal,non précisé,"mariée, mère de 3 enfants",...,non précisé,non précisé,non précisé,A gauche pour faire réussir la France,"Élections législatives de 2017, Ain - 01, circ...",2017-06-11,1,Ain,01,1
1,LG17-1-1-BONNOT-10-tour1-profession_foi.pdf,Bonnot,Gilbert,non précisé,H,Moniteur Educateur,non précisé,non précisé,non précisé,non précisé,...,non précisé,non précisé,non précisé,non précisé,"Élections législatives de 2017, Ain - 01, circ...",2017-06-11,1,Ain,01,1
2,LG17-1-1-BRETON-5-tour1-profession_foi.pdf,Breton,Xavier,54,H,non précisé,conseiller régional,député de la 1ère circonscription de l’Ain dep...,"sports et loisirs, culture","marié, père de 8 enfants",...,non précisé,non précisé,non précisé,non précisé,"Élections législatives de 2017, Ain - 01, circ...",2017-06-11,1,Ain,01,1
3,LG17-1-1-BRETON-5-tour2-profession_foi.pdf,Breton,Xavier,54,H,non précisé,conseiller régional,député de la 1ère circonscription de l’Ain dep...,"sports et loisirs, culture","marié, père de 8 enfants",...,non précisé,non précisé,non précisé,non précisé,"Élections législatives de 2017, Ain - 01, circ...",2017-06-18,2,Ain,01,1
4,LG17-1-1-CARLIER-6-tour1-profession_foi.pdf,Carlier,Pas de candidat individuel,non précisé,NR,non précisé,non précisé,non précisé,non précisé,non précisé,...,non précisé,non précisé,Union populaire républicaine,non précisé,"Élections législatives de 2017, Ain - 01, circ...",2017-06-11,1,Ain,01,1


In [ ]:
#### Pour mettre les informations de tous les batch dans un unique DataFrame, puis dans un unique fichier csv
dfs = []

for i in range(50):
    path = f"/content/drive/MyDrive/Colab Notebooks/Hackathon/SciencesPo/Traitement OCR Fitz/Results/resultsBatch_{i}.csv"
    try:
        df = pd.read_csv(path)
        df = df.dropna(how="all")
        dfs.append(df)
    except FileNotFoundError:
        print(f"Missing file: {path}")

final_df = pd.concat(dfs, ignore_index=True)

print(len(final_df), "rows total")

final_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Hackathon/SciencesPo/Traitement OCR Fitz/Results/resultsAll.csv")

550 rows total
